# Pipecat Cascade + LangSmith

This notebook keeps the workshop-facing pieces visible: the agent setup and the LangSmith tracing setup. The final run cell uses the maintained backend implementation through `workshop`.

## 1. Build The Agent

The cascade has three model-facing stages: speech-to-text, a LangGraph-backed LLM service, and text-to-speech. The graph owns tool use and reasoning; Pipecat owns streaming audio between stages.

In [ ]:
import os
import uuid

from dotenv import load_dotenv
from pipecat.services.openai.stt import OpenAISTTService
from pipecat.services.openai.tts import OpenAITTSService

from voice_demo.pipecat_with_langgraph.graph import GREETING, SYSTEM_PROMPT, build_graph
from voice_demo.pipecat_with_langgraph.langgraph_llm_service import LangGraphLLMService
from workshop import run_pipecat_cascade

load_dotenv()

PROJECT = "voice-workshop-pipecat-cascade"
STT_MODEL = os.getenv("PIPECAT_STT_MODEL", "gpt-4o-mini-transcribe")
LLM_MODEL = os.getenv("PIPECAT_LLM_MODEL", "gpt-4o-mini")
TTS_VOICE = os.getenv("PIPECAT_TTS_VOICE", "alloy")

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook."
assert os.getenv("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY before running this notebook."

In [ ]:
stt = OpenAISTTService(settings=OpenAISTTService.Settings(model=STT_MODEL))

llm = LangGraphLLMService(
    graph=build_graph(SYSTEM_PROMPT),
    settings=LangGraphLLMService.Settings(
        model=LLM_MODEL,
        system_instruction=SYSTEM_PROMPT,
    ),
)

tts = OpenAITTSService(settings=OpenAITTSService.Settings(voice=TTS_VOICE))

## 2. Configure Tracing

The Pipecat integration reads LangSmith settings from the environment. For the workshop, tracing setup is just a fresh thread id plus `configure_pipecat`.

In [ ]:
from langsmith.integrations.pipecat import configure_pipecat, set_thread_id

conversation_id = str(uuid.uuid4())
set_thread_id(conversation_id)

span_processor = configure_pipecat(
    project=PROJECT,
    service_name="workshop-cascade",
    llm_span_kind="chain",
)
assert span_processor is not None

## 3. Put It Together

The traced Pipecat app is a pipeline: local mic input, STT, context aggregation, the LangGraph LLM service, TTS, speaker output, audio recording, and assistant context aggregation.

In [ ]:
from pipecat.audio.vad.silero import SileroVADAnalyzer
from pipecat.pipeline.pipeline import Pipeline
from pipecat.pipeline.task import PipelineParams, PipelineTask
from pipecat.processors.aggregators.llm_context import LLMContext
from pipecat.processors.aggregators.llm_response_universal import (
    LLMContextAggregatorPair,
    LLMUserAggregatorParams,
)
from pipecat.processors.audio.audio_buffer_processor import AudioBufferProcessor
from pipecat.transports.local.audio import LocalAudioTransport, LocalAudioTransportParams

transport = LocalAudioTransport(
    LocalAudioTransportParams(audio_in_enabled=True, audio_out_enabled=True)
)
context = LLMContext()
context_aggregator = LLMContextAggregatorPair(
    context,
    user_params=LLMUserAggregatorParams(vad_analyzer=SileroVADAnalyzer()),
)
audiobuffer = AudioBufferProcessor(num_channels=2, buffer_size=32_000)
span_processor.attach_audio_buffer(audiobuffer, conversation_id)

pipeline = Pipeline(
    [
        transport.input(),
        stt,
        context_aggregator.user(),
        llm,
        tts,
        transport.output(),
        audiobuffer,
        context_aggregator.assistant(),
    ]
)

task = PipelineTask(
    pipeline,
    params=PipelineParams(enable_metrics=True),
    enable_tracing=True,
    enable_turn_tracking=True,
    conversation_id=conversation_id,
)

task

## 4. Run The Agent Live

The cells above show the shape of the implementation. This cell runs the shared backend code so the workshop does not maintain a second Pipecat runner.

In [ ]:
await run_pipecat_cascade(PROJECT)